# LIANA — Heart sn Atlas: Per-Mac Subcluster Interactions

Reload from 4 h5ads → concat → run LIANA → visualisations.

**10 groups in `anno_layer1`:** Cardiomyocyte, EC, Fibroblast, Mac0–Mac6.


## 0. Imports

In [ ]:
import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import liana as li
import networkx as nx
from pathlib import Path


## 1. Paths

In [ ]:
EC_PATH  = '/home/yanjun3/atlas/heart_sn/thesis/ec_20k.h5ad'
FIB_PATH = '/home/yanjun3/atlas/heart_sn/thesis/fb_20k.h5ad'
CM_PATH  = '/home/yanjun3/atlas/heart_sn/thesis/cm_20k.h5ad'
MAC_PATH = '/home/yanjun2/atlas_project/heart_sn_h5ad/sn_final.h5ad'

LAYER   = 'log_counts'
OUT_DIR = Path('/home/yanjun3/atlas/heart_sn/thesis/liana')
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load 4 h5ads, assign anno_layer1 for non-Mac

In [ ]:
ad_ec  = sc.read(EC_PATH)
ad_fib = sc.read(FIB_PATH)
ad_cm  = sc.read(CM_PATH)
ad_mac = sc.read(MAC_PATH)

ad_ec.obs['anno_layer1']  = 'EC'
ad_fib.obs['anno_layer1'] = 'Fibroblast'
ad_cm.obs['anno_layer1']  = 'Cardiomyocyte'

for name, a in [('EC', ad_ec), ('Fib', ad_fib), ('CM', ad_cm), ('Mac', ad_mac)]:
    print(f'{name}: {a.n_obs} cells')
    print(f'  anno_layer1: {a.obs["anno_layer1"].value_counts().to_dict()}')


## 3. Concatenate

In [ ]:
rename_map = {f'Mac{i}': f'heart_sn_Mac{i}' for i in range(7)}
ad_mac.obs['anno_layer1'] = ad_mac.obs['anno_layer1'].astype(str).replace(rename_map)
ad_mac.obs['anno_layer1'] = ad_mac.obs['anno_layer1'].astype('category')

print(ad_mac.obs['anno_layer1'].value_counts())

In [ ]:
adata = anndata.concat(
    [ad_ec, ad_fib, ad_cm, ad_mac],
    join='inner',
    label='source_celltype',
    keys=['EC', 'Fib', 'CM', 'Mac'],
    index_unique='_',
)
print(adata)
print(adata.obs['anno_layer1'].value_counts())


In [ ]:
del ad_ec, ad_fib, ad_cm, ad_mac
import gc
gc.collect()

## 4. Set `.X` to log_counts

In [ ]:
adata.X = adata.layers[LAYER].copy()
print('.X dtype:', adata.X.dtype)
print('.X min / max / mean:', adata.X.min(), adata.X.max(), adata.X.mean())


## 5. Run LIANA consensus

In [ ]:
li.mt.rank_aggregate(
    adata,
    groupby='anno_layer1',
    resource_name='consensus',
    expr_prop=0.1,
    min_cells=5,
    n_perms=1000,
    use_raw=False,
    verbose=True,
)

res = adata.uns['liana_res'].copy()
print('LIANA result shape:', res.shape)
print('Sources:', sorted(res['source'].unique()))
print('Targets:', sorted(res['target'].unique()))


## 6. Rename Mac0–Mac6 → heart_sn_Mac0–Mac6

In [ ]:
non_mac = ['Cardiomyocyte', 'EC', 'Fibroblast']
mac_list = [f'heart_sn_Mac{i}' for i in range(7)]
all_groups = non_mac + mac_list

print('Groups in order:', all_groups)
print('anno_layer1 values:', sorted(adata.obs['anno_layer1'].unique()))
print('LIANA sources:', sorted(res['source'].unique()))

## 7. Score columns + significance filter

In [ ]:
print('Available columns:', res.columns.tolist())

SCORE_COL = 'magnitude_rank'
SIG_COL   = 'specificity_rank'
RANK_THRESHOLD = 0.05

sig = res[res[SIG_COL] < RANK_THRESHOLD].copy()
res['strength'] = -np.log10(res[SCORE_COL].replace(0, np.nan) + 1e-3)

print(f'{len(sig)} significant LR pairs (specificity_rank < {RANK_THRESHOLD})')


## Figure 1 — Pairwise significant LR-pair count heatmap

In [ ]:
interaction_counts = (
    sig.groupby(['source', 'target']).size()
       .unstack(fill_value=0)
       .reindex(index=all_groups, columns=all_groups, fill_value=0)
)
print(interaction_counts)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    interaction_counts, cmap='Reds', square=True,
    linewidths=0.5, linecolor='white',
    annot=True, fmt='d', annot_kws={'size': 8},
    cbar_kws={'label': 'Number of significant LR pairs'},
    ax=ax,
)
ax.set_xlabel('Target (receptor side)')
ax.set_ylabel('Source (ligand side)')
ax.set_title('Pairwise significant LR-pair counts across cell types')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_pairwise_count_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 2 — Mean interaction strength heatmap

In [ ]:
mean_strength = (
    res.groupby(['source', 'target'])['strength'].mean()
       .unstack(fill_value=0)
       .reindex(index=all_groups, columns=all_groups, fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    mean_strength, cmap='RdBu_r', center=mean_strength.values.mean(),
    square=True, linewidths=0.5, linecolor='white',
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cbar_kws={'label': 'Mean −log10(magnitude rank)'},
    ax=ax,
)
ax.set_xlabel('Target')
ax.set_ylabel('Source')
ax.set_title('Mean LR interaction strength (all pairs)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_pairwise_strength_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 3 — Circle network plot

In [ ]:
G = nx.DiGraph()
for src in interaction_counts.index:
    for tgt in interaction_counts.columns:
        w = interaction_counts.loc[src, tgt]
        if w > 0 and src != tgt:
            G.add_edge(src, tgt, weight=int(w))

pos = nx.circular_layout(G)
node_colors = ['#90CAF9' if n in non_mac else '#FFAB40' for n in G.nodes]
node_sizes  = [1000 + 30 * G.degree(n) for n in G.nodes]
max_w = max(d['weight'] for _, _, d in G.edges(data=True))
edge_widths = [4 * d['weight'] / max_w for _, _, d in G.edges(data=True)]

fig, ax = plt.subplots(figsize=(13, 13))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                       edgecolors='black', linewidths=1.5, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='#555',
                       alpha=0.55, arrows=True, arrowsize=14,
                       arrowstyle='-|>', connectionstyle='arc3,rad=0.12',
                       node_size=node_sizes, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold',
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2),
                        ax=ax)
ax.set_title('Cell-cell communication network (edge width = sig. LR count)',
             fontsize=14, pad=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_circle_network.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 4 — Per-Mac outgoing to non-Mac (bar)

In [ ]:
mac_outgoing = (
    res[res['source'].isin(mac_list)]
    .groupby(['source', 'target'])['strength'].mean()
    .unstack()
    .reindex(index=mac_list, columns=all_groups, fill_value=0)
)
mac_outgoing_to_other = mac_outgoing[non_mac]

fig, ax = plt.subplots(figsize=(9, 5))
mac_outgoing_to_other.plot(kind='bar', ax=ax,
                            color=['#EF5350', '#42A5F5', '#66BB6A'],
                            edgecolor='black', linewidth=0.6)
ax.set_ylabel('Mean −log10(magnitude rank)')
ax.set_xlabel('Mac subcluster')
ax.set_title('Mac subcluster → other cell types (outgoing strength)')
ax.legend(title='Target', frameon=False)
plt.xticks(rotation=0)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4_mac_outgoing_to_other.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 5 — Per-Mac incoming from non-Mac (bar)

In [ ]:
mac_incoming = (
    res[res['target'].isin(mac_list)]
    .groupby(['source', 'target'])['strength'].mean()
    .unstack()
    .reindex(index=all_groups, columns=mac_list, fill_value=0)
)
mac_incoming_from_other = mac_incoming.loc[non_mac].T

fig, ax = plt.subplots(figsize=(9, 5))
mac_incoming_from_other.plot(kind='bar', ax=ax,
                              color=['#EF5350', '#42A5F5', '#66BB6A'],
                              edgecolor='black', linewidth=0.6)
ax.set_ylabel('Mean −log10(magnitude rank)')
ax.set_xlabel('Mac subcluster')
ax.set_title('Mac subcluster ← other cell types (incoming strength)')
ax.legend(title='Source', frameon=False)
plt.xticks(rotation=0)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig5_mac_incoming_from_other.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 6 — Outgoing LIANA dotplots per Mac (loop)

In [ ]:
for mac in mac_list:
    try:
        p = li.pl.dotplot(
            adata=adata,
            colour='magnitude_rank',
            size='specificity_rank',
            source_labels=[mac],
            target_labels=non_mac,
            top_n=15,
            orderby='magnitude_rank',
            orderby_ascending=True,
            figure_size=(8, 6),
        )
        p.save(filename=str(OUT_DIR / f'fig6_{mac}_outgoing_dotplot.png'),
               dpi=300, verbose=False)
        print(p)
    except Exception as e:
        print(f'{mac} outgoing dotplot failed:', e)


## Figure 7 — Incoming LIANA dotplots per Mac (loop)

In [ ]:
for mac in mac_list:
    try:
        p = li.pl.dotplot(
            adata=adata,
            colour='magnitude_rank',
            size='specificity_rank',
            source_labels=non_mac,
            target_labels=[mac],
            top_n=15,
            orderby='magnitude_rank',
            orderby_ascending=True,
            figure_size=(8, 6),
        )
        # plotnine returns a ggplot object — save directly
        p.save(filename=str(OUT_DIR / f'fig7_{mac}_incoming_dotplot.png'),
               dpi=300, verbose=False)
        print(p)   # display in jupyter
    except Exception as e:
        print(f'{mac} incoming dotplot failed:', e)

## Figure 8 — Mac-subcluster-specific outgoing LR pairs

In [ ]:
mac_only = res[res['source'].isin(mac_list)].copy()
mac_only['lr_pair'] = (mac_only['ligand_complex'].astype(str) + '→' +
                       mac_only['receptor_complex'].astype(str))

pivot = mac_only.pivot_table(
    index=['lr_pair', 'target'],
    columns='source',
    values='strength',
    aggfunc='max',
).reindex(columns=mac_list)
pivot = pivot.dropna(how='all')

top_mac = pivot.idxmax(axis=1)
max_strength = pivot.max(axis=1)
second_strength = pivot.apply(
    lambda r: sorted(r.dropna())[-2] if r.dropna().shape[0] >= 2 else 0, axis=1
)
specificity_gap = max_strength - second_strength

ranking = pd.DataFrame({
    'top_mac': top_mac,
    'max_strength': max_strength,
    'gap': specificity_gap,
}).sort_values('gap', ascending=False)
print(ranking.head(25))


In [ ]:
top_pairs = ranking.head(25).index
top_pivot = pivot.loc[top_pairs]
top_pivot.index = [f'{lr}  ({tgt})' for lr, tgt in top_pivot.index]

fig, ax = plt.subplots(figsize=(8, max(6, len(top_pivot) * 0.3)))
sns.heatmap(
    top_pivot.fillna(0), cmap='Reds', linewidths=0.3,
    cbar_kws={'label': '−log10(magnitude rank)'}, ax=ax,
)
ax.set_title('Mac-subcluster-specific outgoing LR pairs (top 25 by specificity gap)',
             fontsize=12)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig8_mac_specific_outgoing_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 9 — Prior pathways across cell-cell edges

In [ ]:
priors = [
    ('IL33', 'IL1RL1'),
    ('CSF1', 'CSF1R'),
    ('IL34', 'CSF1R'),
    ('IGF1', 'IGF1R'),
    ('PDGFC', 'PDGFRA'),
    ('VEGFC', 'FLT4'),
    ('ANGPT1', 'TEK'),
    ('CXCL12', 'CXCR4'),
    ('HBEGF', 'EGFR'),
]

def lr_match(row, l, r):
    return (l in str(row.get('ligand_complex', ''))) and \
           (r in str(row.get('receptor_complex', '')))

prior_hits = []
for l, r in priors:
    hit = res[res.apply(lambda x: lr_match(x, l, r), axis=1)].copy()
    if not hit.empty:
        hit['lr_pair'] = f'{l}→{r}'
        prior_hits.append(hit)

prior_df = pd.concat(prior_hits, ignore_index=True) if prior_hits else pd.DataFrame()
print(f'Found {len(prior_df)} prior-pathway interactions')
prior_df.head()


In [ ]:
if not prior_df.empty:
    prior_df['edge'] = prior_df['source'] + ' → ' + prior_df['target']
    pivot_prior = prior_df.pivot_table(
        index='lr_pair', columns='edge', values='strength', aggfunc='max',
    )

    fig, ax = plt.subplots(figsize=(max(10, pivot_prior.shape[1] * 0.35), 6))
    sns.heatmap(
        pivot_prior.fillna(0), cmap='RdBu_r',
        center=pivot_prior.values.mean(),
        linewidths=0.3,
        cbar_kws={'label': '−log10(magnitude rank)'},
        ax=ax,
    )
    ax.set_title('Prior pathway LR pairs across cell-cell edges')
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'fig9_prior_pathway_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()


## Figure 10 — Mac0–3 outgoing LR interactions to non-Mac (unified heatmap)

Direct comparison: rows = shared top LR pairs across Mac0–3, columns = `Mac × Target` (4 × 3 = 12 columns), one color scale.
Exposes how each Mac sends signals to CM / EC / Fibroblast side-by-side. Mac0/Mac3 = real LYVE1+ residents; Mac1 (Fib-doublet) and Mac2 (CM-doublet) outgoing reflects ambient contamination.


In [ ]:
macs_focus = [f'heart_sn_Mac{i}' for i in range(4)]
others = ['Cardiomyocyte', 'EC', 'Fibroblast']

df = res.query('source in @macs_focus and target in @others').copy()
df['lr_pair'] = df['ligand_complex'].astype(str) + ' → ' + df['receptor_complex'].astype(str)
df['edge'] = df['source'].str.replace('heart_sn_', '') + ' → ' + df['target']
df['score'] = -np.log10(df['magnitude_rank'].clip(lower=1e-3))

top_pairs = (df.groupby('lr_pair')['score'].max()
             .sort_values(ascending=False).head(30).index.tolist())

mat = (df.query('lr_pair in @top_pairs')
       .pivot_table(index='lr_pair', columns='edge', values='score', fill_value=0))
col_order = [f'Mac{i} → {t}' for i in range(4) for t in others]
mat = mat.reindex(columns=col_order)

g = sns.clustermap(
    mat, cmap='Reds', figsize=(7, 11),
    row_cluster=True, col_cluster=False,
    linewidths=0.4, linecolor='white',
    dendrogram_ratio=(0.08, 0),
    cbar_pos=(1.02, 0.4, 0.03, 0.25),
    cbar_kws={'label': '-log10(magnitude_rank)'},
    xticklabels=True, yticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=60, ha='right')
for x in [3, 6, 9]:
    g.ax_heatmap.axvline(x, color='black', linewidth=1.2)
g.fig.suptitle('Mac0–3 outgoing LR interactions to non-Mac cell types',
               y=1.02, fontsize=12)
plt.savefig(OUT_DIR / 'fig10_mac0_3_outgoing_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 11 — Mac0–3 incoming LR interactions from non-Mac (unified heatmap)

Mirror of Fig 10. Rows = shared top LR pairs, columns = `Source → Mac` (3 × 4 = 12). Reveals who's signaling TO each Mac subcluster (expect Fib-heavy ECM, EC angiopoietins).


In [ ]:
df_in = res.query('source in @others and target in @macs_focus').copy()
df_in['lr_pair'] = df_in['ligand_complex'].astype(str) + ' → ' + df_in['receptor_complex'].astype(str)
df_in['edge'] = df_in['source'] + ' → ' + df_in['target'].str.replace('heart_sn_', '')
df_in['score'] = -np.log10(df_in['magnitude_rank'].clip(lower=1e-3))

top_pairs_in = (df_in.groupby('lr_pair')['score'].max()
                .sort_values(ascending=False).head(30).index.tolist())

mat_in = (df_in.query('lr_pair in @top_pairs_in')
          .pivot_table(index='lr_pair', columns='edge', values='score', fill_value=0))
col_order_in = [f'{s} → Mac{i}' for i in range(4) for s in others]
mat_in = mat_in.reindex(columns=col_order_in)

g = sns.clustermap(
    mat_in, cmap='Blues', figsize=(7, 11),
    row_cluster=True, col_cluster=False,
    linewidths=0.4, linecolor='white',
    dendrogram_ratio=(0.08, 0),
    cbar_pos=(1.02, 0.4, 0.03, 0.25),
    cbar_kws={'label': '-log10(magnitude_rank)'},
    xticklabels=True, yticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=60, ha='right')
for x in [3, 6, 9]:
    g.ax_heatmap.axvline(x, color='black', linewidth=1.2)
g.fig.suptitle('Mac0–3 incoming LR interactions from non-Mac cell types',
               y=1.02, fontsize=12)
plt.savefig(OUT_DIR / 'fig11_mac0_3_incoming_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 12 — Mac0 vs Mac3 only (real LYVE1+ residents)

Drops the doublet-contaminated Mac1/Mac2. Direct contrast of the two clean LYVE1+ resident states (Mac0 MHC-II+ antigen-presenting vs Mac3 motile/migratory) against CM / EC / Fib targets.


In [ ]:
macs_clean = ['heart_sn_Mac0', 'heart_sn_Mac3']

df_clean = res.query('source in @macs_clean and target in @others').copy()
df_clean['lr_pair'] = df_clean['ligand_complex'].astype(str) + ' → ' + df_clean['receptor_complex'].astype(str)
df_clean['edge'] = df_clean['source'].str.replace('heart_sn_', '') + ' → ' + df_clean['target']
df_clean['score'] = -np.log10(df_clean['magnitude_rank'].clip(lower=1e-3))

top_clean = (df_clean.groupby('lr_pair')['score'].max()
             .sort_values(ascending=False).head(25).index.tolist())

mat_clean = (df_clean.query('lr_pair in @top_clean')
             .pivot_table(index='lr_pair', columns='edge', values='score', fill_value=0))
col_order_clean = [f'{m} → {t}' for m in ['Mac0', 'Mac3'] for t in others]
mat_clean = mat_clean.reindex(columns=col_order_clean)

g = sns.clustermap(
    mat_clean, cmap='Reds', figsize=(5.5, 9.5),
    row_cluster=True, col_cluster=False,
    linewidths=0.4, linecolor='white',
    dendrogram_ratio=(0.08, 0),
    cbar_pos=(1.02, 0.4, 0.03, 0.25),
    cbar_kws={'label': '-log10(magnitude_rank)'},
    xticklabels=True, yticklabels=True,
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=60, ha='right')
g.ax_heatmap.axvline(3, color='black', linewidth=1.5)
g.fig.suptitle('Mac0 vs Mac3 (LYVE1+ residents) → non-Mac',
               y=1.02, fontsize=12)
plt.savefig(OUT_DIR / 'fig12_mac0_mac3_clean_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 13 — Mac-specificity scores: Mac0/Mac3 vs Mac1/Mac2 contrast

For each LR pair Mac → non-Mac, computes (mean score in clean residents [Mac0+Mac3]) − (mean score in doublet-suspected [Mac1+Mac2]). Positive (red) = real-resident-specific signal; negative (blue) = doublet-driven. Direct evidence flagging the contamination story.


In [ ]:
mat_all = mat.copy()
clean_cols = [c for c in mat_all.columns if 'Mac0' in c or 'Mac3' in c]
doublet_cols = [c for c in mat_all.columns if 'Mac1' in c or 'Mac2' in c]

contrast = pd.DataFrame({
    'Cardiomyocyte': mat_all[[c for c in clean_cols if 'Cardio' in c]].mean(axis=1) -
                     mat_all[[c for c in doublet_cols if 'Cardio' in c]].mean(axis=1),
    'EC':            mat_all[[c for c in clean_cols if 'EC' in c]].mean(axis=1) -
                     mat_all[[c for c in doublet_cols if 'EC' in c]].mean(axis=1),
    'Fibroblast':    mat_all[[c for c in clean_cols if 'Fibroblast' in c]].mean(axis=1) -
                     mat_all[[c for c in doublet_cols if 'Fibroblast' in c]].mean(axis=1),
})

vmax = contrast.abs().max().max()
fig, ax = plt.subplots(figsize=(4.5, 10))
sns.heatmap(
    contrast.sort_values('Fibroblast', ascending=False),
    cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Δ score (clean − doublet)'}, ax=ax,
)
ax.set_xlabel('Target')
ax.set_title('LR pair specificity:\nMac0/Mac3 (clean) vs Mac1/Mac2 (doublet-suspected)',
             fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig13_clean_vs_doublet_contrast.png', dpi=300, bbox_inches='tight')
plt.show()


## Save outputs

In [ ]:
anndata.settings.allow_write_nullable_strings = True

res.to_csv(OUT_DIR / 'liana_full_results_relabeled.csv', index=False)
sig.to_csv(OUT_DIR / 'liana_significant_results.csv', index=False)
ranking.to_csv(OUT_DIR / 'mac_specific_outgoing_ranking.csv')

adata.write(OUT_DIR / 'heart_sn_liana_relabeled.h5ad')
print('Saved everything to', OUT_DIR)


In [ ]:
# Check actual column names
print(res.columns.tolist())

# Check if IL33 / IL1RL1 exist at all in the result
print('IL33 sources:', res[res['ligand_complex'].astype(str).str.contains('IL33', na=False)]['ligand_complex'].unique())
print('IL1RL1 receptors:', res[res['receptor_complex'].astype(str).str.contains('IL1RL1', na=False)]['receptor_complex'].unique())

# Check all unique ligand names
print('Total unique ligands:', res['ligand_complex'].nunique())
print('Sample ligands:', res['ligand_complex'].sample(20).tolist())